# D6 · October tuning and frozen November evaluation

Start with **plan** on CPU. It displays the registered configurations and saves their hashes. For training, choose one model and **all** to run the eight seed-42 configurations and confirm the two finalists. MF and LightGBM use CPU. Use an NVIDIA GPU for either sequence model in Colab. Run **freeze** once after all four models finish. Then choose **test** once for each model.

# D6 · 十月调参与冻结后的十一月评估

首次在 CPU 选择 **plan**，查看预登记配置并保存校验值。训练时选一个模型和 **all**，程序先以种子 42 比较 8 组配置，再对前两组补跑另外两个种子。MF、LightGBM 用 CPU，两个序列模型在 Colab 选 NVIDIA GPU。四个模型都完成后运行一次 **freeze**，再为每个模型各运行一次 **test**。

In [1]:
#@title Experiment controls / 实验选项
STAGE = "plan" #@param ["plan", "all", "screen", "confirm", "freeze", "test"]
MODEL = "lrurec" #@param ["mf", "lightgbm", "lrurec", "mamba4rec"]
DRIVE_FOLDER = "CS5481_Project2_OnlineRetail" #@param {type:"string"}
REUSE_BASELINE = True #@param {type:"boolean"}
BASELINE_RUN = "" #@param {type:"string"}

## Source and result locations

The notebook reads D2 from a pinned Git checkout. Colab saves weights, receipts and score tables under the project’s **results/** in My Drive. Each notebook session also has its own log. For baseline reuse, leave **BASELINE_RUN** blank to check prior Drive sessions in date order, or enter the path to a specific session. A qualifying run needs the original checkpoint, full October scores and matching configuration. To measure trials 01 and 08 on a new GPU under identical conditions, clear **REUSE_BASELINE** before starting the model’s first search.

## 源码和结果位置

本 notebook 读取固定 Git 提交中的 D2 数据。Colab 把权重、运行记录和成绩表写入「我的云端硬盘」项目的 **results/**，并为每次 notebook 会话保存日志。**BASELINE_RUN** 留空时，程序按时间从早到晚核验 Drive 中已有的 baseline。也可以填入指定会话的路径。复用需要原始权重、完整十月评分及匹配的配置。若要在新 GPU 上同环境比较 01 和 08，可在该模型首次搜索前取消勾选 **REUSE_BASELINE**。

In [2]:
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import platform
import shlex
import subprocess
import sys
import tempfile
import uuid

try:
    from google.colab import drive
except ImportError:
    drive = None
on_colab = drive is not None
runtime = "Google Colab" if on_colab else "local Jupyter"
revision = "989bea5dd354246aa179f745f39fe4a03669e3aa"
if on_colab:
    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    source = "https://github.com/NobunagaK/online-retail-top10-recommendation.git"
    cache = Path("/content/retail_d6_code")
else:
    start = Path(os.environ.get("RETAIL_SOURCE_ROOT", Path.cwd())).resolve()
    repo = next(p for p in (start, *start.parents) if (p / ".git").exists())
    project = Path(os.environ.get("RETAIL_D6_PROJECT", repo))
    source = str(repo)
    cache = Path(tempfile.gettempdir()) / "retail_d6_code"
checkout = cache / revision
checkout.mkdir(parents=True, exist_ok=True)
def git(*args):
    return subprocess.check_output(["git", "-C", str(checkout), *args], text=True).strip()
if not (checkout / ".git").exists():
    git("init", "--quiet")
if subprocess.run(["git", "-C", str(checkout), "cat-file", "-e", revision + "^{commit}"], capture_output=True).returncode:
    git("fetch", "--quiet", "--depth=1", source, revision)
git("-c", "advice.detachedHead=false", "checkout", "--quiet", "--detach", revision)
if git("rev-parse", "HEAD") != revision or git("status", "--porcelain", "--untracked-files=no"):
    raise RuntimeError("The checkout differs from the pinned D6 source.")
results = project / "results"
data = checkout / "data/processed/d2"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
session = project / "experiments/d6_sessions" / f"{MODEL}_{STAGE}_{stamp}_{uuid.uuid4().hex[:8]}"
session.mkdir(parents=True)
transcript = session / "session.log"

def log(message):
    print(message, flush=True)
    with transcript.open("a", encoding="utf-8") as out:
        print(message, file=out)

def run(command, *, label=None, show=True):
    command = [str(x) for x in command]
    log(label or "$ " + shlex.join(command))
    with transcript.open("a", encoding="utf-8") as out, subprocess.Popen(
        command, cwd=checkout, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1
    ) as process:
        for line in process.stdout:
            if show:
                print(line, end="", flush=True)
            out.write(line)
            out.flush()
        if process.wait():
            raise RuntimeError(f"Command failed. See {transcript}")
log(f"Source {revision} | {runtime} | {STAGE} | {MODEL}")
log(f"Results: {results}")

Source 989bea5dd354246aa179f745f39fe4a03669e3aa | local Jupyter | plan | lrurec


Results: /tmp/retail-d6-notebook-check/results


## Model environment

The next cell uses the **environment** and **preflight** cells from the pinned pilot notebook. This gives Mamba4Rec the same CUDA 12.6 / PyTorch 2.7.0 environment and native gradient check used for its T4 pilot. The plan and freeze stages use CPU.

## 模型环境

下方直接使用固定版本试训 notebook 的 **environment** 和 **preflight** 单元格。Mamba4Rec 沿用 T4 试训验证过的 CUDA 12.6、PyTorch 2.7.0 和原生算子梯度检查。登记和冻结阶段只需 CPU。

In [3]:
admin_stage = STAGE in {"plan", "freeze"}
setup_model = "lightgbm" if admin_stage else MODEL
device = "cuda" if setup_model == "mamba4rec" or (setup_model == "lrurec" and on_colab) else "cpu"
pilot = json.loads((checkout / "notebooks/model_pilots.ipynb").read_text())
pilot_cells = {c["id"]: "".join(c["source"]) for c in pilot["cells"] if c["cell_type"] == "code"}
scope = {**globals(), "MODEL": setup_model}
for cell_id in (["environment"] if admin_stage else ["environment", "preflight"]):
    exec(compile(pilot_cells[cell_id], f"model_pilots.ipynb/{cell_id}", "exec"), scope)
python = scope["python"]

Model Python: /workspace/online-retail-top10-recommendation/.venv/bin/python


## Run the selected stage

**screen** and **confirm** split the work done by **all** across sessions. Repeating a training stage verifies completed trials and resumes interrupted fits at a saved epoch. Use one notebook session per model at a time. The freeze records the selected checkpoint hashes for all three seeds of each model. Test scores appear in the shared table when all four models have finished scoring November.

## 运行所选阶段

**screen** 和 **confirm** 可以分两次完成 **all** 的工作。重新执行训练阶段时，程序核验已完成的组合，从已保存的轮次续跑中断的组合。同一时刻每个模型用一个会话。冻结文件记录四个模型各自三个种子的权重校验值。四个模型都完成十一月评分后，才生成总测试表。

In [4]:
if STAGE in {"plan", "all", "screen", "confirm"}:
    command = [python, "-u", "-m", "scripts.tune", "--stage", STAGE, "--model", MODEL, "--device", device]
    if REUSE_BASELINE:
        command.extend(["--pilots-dir", project / "experiments/model_pilots"])
        if BASELINE_RUN.strip():
            command.extend(["--baseline-run", BASELINE_RUN.strip()])
elif STAGE == "freeze":
    command = [python, "-u", "-m", "scripts.freeze_experiment"]
elif STAGE == "test":
    command = [python, "-u", "-m", "scripts.evaluate", "--model", MODEL, "--device", device, "--project-root", checkout]
else:
    raise ValueError(f"Unknown D6 stage {STAGE}")
run([*command, "--results-dir", results, "--data-dir", data], show=STAGE != "plan")

if STAGE == "plan":
    from IPython.display import display
    import pandas as pd
    registration = json.loads((results / "d6_registration.json").read_text())
    plan = registration["plan"]
    print("October queries:", registration["development"]["validation_queries"],
          "| catalog:", registration["development"]["validation_candidates"],
          "| configurations per model:", {m: len(rows) for m, rows in plan["configs"].items()})
    print("Seeds:", plan["seeds"], "| finalists per model:", plan["finalists"])
    table = pd.DataFrame.from_dict(plan["configs"][MODEL], orient="index")
    if MODEL in {"lrurec", "mamba4rec"}:
        table["optimizer"] = table["optimizer"].fillna("adam")
        table["lr_scheduler"] = table["lr_scheduler"].fillna("none")
    display(table.rename_axis("config_id"))
else:
    for name in ["tuning.csv", "validation_metrics.csv", "selection.json", "test_metrics.csv"]:
        if (results / name).is_file():
            print(results / name)

$ /workspace/online-retail-top10-recommendation/.venv/bin/python -u -m scripts.tune --stage plan --model lrurec --device cpu --pilots-dir /tmp/retail-d6-notebook-check/experiments/model_pilots --results-dir /tmp/retail-d6-notebook-check/results --data-dir /tmp/retail_d6_code/989bea5dd354246aa179f745f39fe4a03669e3aa/data/processed/d2


October queries: 758 | catalog: 3495 | configurations per model: {'mf': 8, 'lightgbm': 8, 'lrurec': 8, 'mamba4rec': 8}
Seeds: [42, 2026, 3407] | finalists per model: 2


,seed,epochs,patience,max_length,hidden_dim,num_layers,dropout,batch_size,learning_rate,weight_decay,optimizer,lr_scheduler
config_id,,,,,,,,,,,,
01,42,100,10,64,64,2,0.2,64,0.0010,0.0100,adam,none
02,42,100,15,64,64,2,0.2,64,0.0010,0.0100,adamw,none
03,42,100,15,128,64,2,0.2,64,0.0010,0.0100,adamw,none
04,42,100,15,256,64,2,0.2,64,0.0010,0.0100,adamw,none
05,42,100,15,128,64,2,0.2,64,0.0010,0.0100,adamw,cosine
06,42,100,15,128,128,2,0.1,64,0.0010,0.0001,adamw,cosine
07,42,100,15,256,128,2,0.3,64,0.0003,0.0000,adamw,cosine
08,42,100,10,64,64,2,0.2,64,0.0010,0.0100,adamw,cosine
